# Inicio

## Creacion de la sesion de spark

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

## Lectura Batch con AutoLoader


In [0]:


df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", "s3://aws-data-lake-databricks/schema-test/spotify/")
    .load("s3://aws-data-lake-databricks/datasets-test/")
)



## Se agregan nuevos meta datos para obtener trazabilidad y data linage

In [0]:
from pyspark.sql.functions import current_timestamp, col

# 👇 Agregar metadata
df = (df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

## Se escribe en el dataframe en la base de datos bronze

In [0]:
(df.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://aws-data-lake-databricks/checkpoints/spotify_bronze-test/")
    .trigger(availableNow=True) # 👈 modo streaming
    .toTable("dev_workspace.bronze.raw_spotify_tracks")
)

In [0]:
%sql
select *
from dev_workspace.bronze.raw_spotify_tracks

In [0]:
%sql
DROP TABLE dev_workspace.bronze.raw_spotify_tracks